## Phase 3: Convert Extracted Phrases to Feature Scores

In [ ]:
# Phase 3: Convert Extracted Phrases to Feature Scores
import os
import pandas as pd
import numpy as np

# 1) Load df_final_with_extracted_features.csv
current_dir = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
output_dir = os.path.join(current_dir, "output")
input_file = os.path.join(output_dir, "df_final_with_extracted_features.csv")
df = pd.read_csv(input_file)
print("Loaded:", input_file)
print(df[["luxury_features", "transport_mentions", "school_mentions", "renovation_mentions"]].head())

# 2) Define rules to transfer phrases into scores

# Function: check if cell has any non-empty text
def has_text(x):
    if isinstance(x, str):
        return x.strip() != ""
    return False

# 2.1 luxury_features: 3 levels (0, 1, 2)
# Example rule:
# 0 = no luxury_features text
# 1 = some simple luxury phrase (short list / few amenities)
# 2 = many or very strong luxury amenities (long list, multiple features)
def score_luxury(text):
    if not isinstance(text, str) or text.strip() == "":
        return 0

    t = text.lower()
    # simple heuristic based on length and keyword count, using observed patterns in sample rows [file:3]
    luxury_keywords = [
        'security', "swimming pool", "pool", "spa", "gym", "cinema", "media room",
        "games room", "wine cellar", "staff accommodation", "sauna", "steam room",
        "landscaped", "private rear garden", "luxurious", "state-of-the-art",'garden',
        'terrace', 'wine room', 'jacuzzi','treatment room', 'home cinema', 'gymnasium',
        'tennis court', 'billiards', 'bar', 'concierge', 'valet'
    ]
    kw_count = sum(1 for k in luxury_keywords if k in t)

    # length of phrase and number of amenities both indicate higher luxury
    if kw_count >= 8 or len(t) > 160:
        return 2
    elif kw_count >= 1:
        return 1
    else:
        return 1  # has some text but no keyword match → treat as moderate

# 2.2 transport_mentions: Binary scores (0,1) 
def score_transport(text):
    """Convert transport_mentions text to binary score (0 or 1)"""
    if pd.isna(text) or str(text).strip() == '':
        return 0
    
    text_lower = str(text).lower()
    
    # Check for transport-related keywords
    transport_keywords = [
        'station', 'underground', 'tube', 'transport', 'link', 'access',
        'line', 'rail', 'metro', 'bus', 'road', 'street', 'avenue',
        'quick access', 'easy access', 'walk', 'minute', 'mile'
    ]
    
    for keyword in transport_keywords:
        if keyword in text_lower:
            return 1
    
    return 0

# 2.3 school_mentions: Binary scores (0,1)
def score_school(text):
    """Convert school_mentions text to binary score (0 or 1)"""
    if pd.isna(text) or str(text).strip() == '':
        return 0
    
    text_lower = str(text).lower()
    
    # Check for school-related keywords
    school_keywords = [
        'school', 'college', 'university', 'academy', 'institute',
        'education', 'excellent schools', 'good schools', 'primary',
        'secondary', 'high school', 'grammar', 'private', 'public',
        'catchment', 'schools', 'educational'
    ]
    
    for keyword in school_keywords:
        if keyword in text_lower:
            return 1
    
    return 0

# 2.4 renovation_mentions: Binary scores (0,1)
def score_renovation(text):
    """Convert renovation_mentions text to binary score (0 or 1)"""
    if pd.isna(text) or str(text).strip() == '':
        return 0
    
    text_lower = str(text).lower()
    
    # Check for renovation-related keywords
    renovation_keywords = [
        'renovated', 'refurbished', 'restored', 'redeveloped', 'rebuilt',
        'modernised', 'upgraded', 'refitted', 'renewed', 'reconditioned',
        'reconstructed', 'newly', 'recently', 'turnkey', 'refurbishment',
        'restoration', 'renovation', 'comprehensively', 'extensively',
        'meticulously', 'beautifully', 'expertly', 'thoughtfully'
    ]
    
    for keyword in renovation_keywords:
        if keyword in text_lower:
            return 1
    
    return 0


# 3) Apply scoring and drop original phrase columns
df["luxury_score"] = df["luxury_features"].apply(score_luxury)
df["transport_score"] = df["transport_mentions"].apply(score_transport)
df["school_score"] = df["school_mentions"].apply(score_school)
df["renovation_score"] = df["renovation_mentions"].apply(score_renovation)

# Remove old phrase columns
df = df.drop(columns=["luxury_features", "transport_mentions", "school_mentions", "renovation_mentions"])

# 4) Save as a new DataFrame/file
output_file = os.path.join(output_dir, "df_with_extracted_scores.csv")
df.to_csv(output_file, index=False)
print("Saved with scores to:", output_file)

# Optional: quick sanity check
print(df[["price", "luxury_score", "transport_score", "school_score", "renovation_score"]].head())


Loaded: c:\Users\Admin\Python\S8_Thesis\llm\output\df_final_with_extracted_features.csv
                                     luxury_features  \
0  luxurious and expansive entertaining areas, st...   
1  swimming pool, cinema room and separate staff ...   
2                                                NaN   
3  luxurious and spacious accommodation for enter...   
4                                                NaN   

                                  transport_mentions    school_mentions  \
0          quick access to the City and the West End  excellent schools   
1  just off Park Lane and a stone's throw away fr...                NaN   
2                                                NaN                NaN   
3                                                NaN                NaN   
4                                                NaN                NaN   

  renovation_mentions  
0                 NaN  
1                 NaN  
2                 NaN  
3                 NaN  
4   

## Phase 4: Train & Evaluate Random Forest Models

 - Baseline: df_cleaned.csv (no extracted scores)
 - Enhanced: df_with_extracted_scores.csv (with extracted scores)

In [ ]:
# Phase 4: Train & Evaluate Random Forest Models

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib

# -------------------------------------------------------------------
# Function: evaluate model
# -------------------------------------------------------------------
def evaluate_model(model, X_train, X_test, y_train, y_test, label="model"):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    print(f"\n=== {label} ===")
    print(f"RMSE: {rmse:,.2f}")
    print(f"MAE : {mae:,.2f}")
    return rmse, mae

# -------------------------------------------------------------------
# 1) Baseline model on df_cleaned.csv
# -------------------------------------------------------------------
cleaned_file = os.path.join(output_dir, "df_cleaned.csv")
df_cleaned = pd.read_csv(cleaned_file)
print("Loaded cleaned data:", cleaned_file)
print("Cleaned columns:", df_cleaned.columns.tolist())

target_col = "price"

# numeric features
num_cols = ["sizeSqFeetMax", "bedrooms", "bathrooms"]
# categorical to encode
cat_cols = ["propertyType", "listingUpdateReason"]

# one‑hot encode categoricals
df_cleaned_cat = pd.get_dummies(df_cleaned[cat_cols], prefix=cat_cols, drop_first=False)
X_base = pd.concat([df_cleaned[num_cols].reset_index(drop=True),
                    df_cleaned_cat.reset_index(drop=True)], axis=1)
y_base = df_cleaned[target_col].copy()

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42
)

rf_base = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rmse_base, mae_base = evaluate_model(
    rf_base, Xb_train, Xb_test, yb_train, yb_test,
    label="Random Forest (baseline: num + encoded propertyType/listingUpdateReason)"
)

baseline_model_path = os.path.join(output_dir, "rf_baseline_model.pkl")
joblib.dump(rf_base, baseline_model_path)
print("Baseline model saved to:", baseline_model_path)

# -------------------------------------------------------------------
# 2) Enhanced model on df_with_extracted_scores.csv
#    (must already contain luxury_score, transport_score, school_score, renovation_score)
# -------------------------------------------------------------------
scores_file = os.path.join(output_dir, "df_with_extracted_scores.csv")
df_scores = pd.read_csv(scores_file)
print("\nLoaded data with extracted scores:", scores_file)
print("Columns:", df_scores.columns.tolist())

# numeric + scores
score_cols = ["luxury_score", "transport_score", "school_score", "renovation_score"]
num_cols_scores = ["sizeSqFeetMax", "bedrooms", "bathrooms"] + score_cols

# one‑hot encode same categoricals
df_scores_cat = pd.get_dummies(df_scores[cat_cols], prefix=cat_cols, drop_first=False)

X_scores = pd.concat([df_scores[num_cols_scores].reset_index(drop=True),
                      df_scores_cat.reset_index(drop=True)], axis=1)
y_scores = df_scores[target_col].copy()

Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_scores, y_scores, test_size=0.2, random_state=42
)

rf_scores = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rmse_scores, mae_scores = evaluate_model(
    rf_scores, Xs_train, Xs_test, ys_train, ys_test,
    label="Random Forest (num + encoded categoricals + extracted scores)"
)

scores_model_path = os.path.join(output_dir, "rf_with_scores_model.pkl")
joblib.dump(rf_scores, scores_model_path)
print("Model with scores saved to:", scores_model_path)

# -------------------------------------------------------------------
# 3) Compare performance
# -------------------------------------------------------------------
print("\n=== Performance comparison (test set) ===")
print(f"Baseline RF - RMSE: {rmse_base:,.2f}, MAE: {mae_base:,.2f}")
print(f"With scores RF - RMSE: {rmse_scores:,.2f}, MAE: {mae_scores:,.2f}")

if rmse_scores < rmse_base:
    print("→ RMSE improved after adding extracted feature scores.")
else:
    print("→ RMSE did not improve after adding extracted feature scores.")

if mae_scores < mae_base:
    print("→ MAE improved after adding extracted feature scores.")
else:
    print("→ MAE did not improve after adding extracted feature scores.")


Loaded cleaned data: c:\Users\Admin\Python\S8_Thesis\llm\output\df_cleaned.csv
Cleaned columns: ['title', 'propertyType', 'sizeSqFeetMax', 'bedrooms', 'bathrooms', 'listingUpdateReason', 'price', 'Date', 'listingDescription']

=== Random Forest (baseline: num + encoded propertyType/listingUpdateReason) ===
RMSE: 7,440,453.73
MAE : 5,773,378.85
Baseline model saved to: c:\Users\Admin\Python\S8_Thesis\llm\output\rf_baseline_model.pkl

Loaded data with extracted scores: c:\Users\Admin\Python\S8_Thesis\llm\output\df_with_extracted_scores.csv
Columns: ['title', 'propertyType', 'sizeSqFeetMax', 'bedrooms', 'bathrooms', 'listingUpdateReason', 'price', 'Date', 'listingDescription', 'luxury_score', 'transport_score', 'school_score', 'renovation_score']

=== Random Forest (num + encoded categoricals + extracted scores) ===
RMSE: 7,553,304.38
MAE : 5,856,554.04
Model with scores saved to: c:\Users\Admin\Python\S8_Thesis\llm\output\rf_with_scores_model.pkl

=== Performance comparison (test set) ==